In [ ]:
"""
verify_orbits.py

Independent verification of the noiseless Earth and Mars orbital models
used in Homework 1.1.

Checks:
    1. Specific orbital energy conservation
    2. Specific angular momentum conservation
    3. Agreement with theoretical Kepler values

The verification does NOT depend on the synthetic-observation program.
"""

import math
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. PHYSICAL CONSTANTS
# ============================================================

# Solar gravitational parameter in AU^3/day^2.
#
# This normalization makes an orbit with a = 1 AU have
# approximately Earth's sidereal period of 365.256 days.
MU_SUN = 4 * math.pi**2 / (365.256363004**2)


# ============================================================
# 2. ORBIT PARAMETERS
# ============================================================
#
# These must match the hidden model used to generate the
# synthetic observations.
#
# a      = semi-major axis in AU
# e      = eccentricity
# M0     = mean anomaly at J2000
# varpi  = direction of perihelion in the orbital plane
# ============================================================

EARTH = {
    "a": 1.00000011,
    "e": 0.01671022,
    "M0": math.radians(357.51716),
    "varpi": math.radians(102.93735),
}

MARS = {
    "a": 1.52366231,
    "e": 0.09341233,
    "M0": math.radians(19.41248),
    "varpi": math.radians(336.04084),
}


# ============================================================
# 3. DATE CONVERSION
# ============================================================

def days_from_j2000(dt):
    """
    Return the number of days between dt and the J2000 epoch.

    J2000 epoch:
        January 1, 2000 at 12:00 UTC.
    """

    j2000 = datetime(
        2000,
        1,
        1,
        12,
        tzinfo=timezone.utc
    )

    seconds = (dt - j2000).total_seconds()

    return seconds / 86400.0


# ============================================================
# 4. SOLVE KEPLER'S EQUATION
# ============================================================

def solve_kepler(M, e, tol=1e-13, max_iter=100):
    """
    Solve Kepler's equation

        M = E - e sin(E)

    for the eccentric anomaly E.

    Newton's method is used.
    """

    # Reduce the angle to improve numerical stability.
    M_reduced = (
        (M + math.pi) % (2 * math.pi)
    ) - math.pi

    # Initial guess
    E = M_reduced

    for _ in range(max_iter):

        f = E - e * math.sin(E) - M_reduced

        df = 1 - e * math.cos(E)

        correction = f / df

        E -= correction

        if abs(correction) < tol:
            return E

    raise RuntimeError(
        "Kepler solver failed to converge."
    )


# ============================================================
# 5. ROTATION MATRIX
# ============================================================

def rotate_z(vector, angle):
    """
    Rotate a vector about the z-axis.

    This simply changes the orientation of the ellipse
    without changing its shape, energy, or angular momentum.
    """

    c = math.cos(angle)
    s = math.sin(angle)

    rotation_matrix = np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0]
    ])

    return rotation_matrix @ vector


# ============================================================
# 6. KEPLER ORBIT STATE
# ============================================================

def kepler_state(t_days, params):
    """
    Compute heliocentric position and velocity.

    Input
    -----
    t_days:
        Time measured in days from J2000.

    params:
        Dictionary containing a, e, M0, and varpi.

    Returns
    -------
    r:
        Position vector in AU.

    v:
        Velocity vector in AU/day.
    """

    a = params["a"]
    e = params["e"]
    M0 = params["M0"]
    varpi = params["varpi"]

    # --------------------------------------------------------
    # Mean motion
    #
    # From Kepler/Newton:
    #
    #       n = sqrt(mu / a^3)
    #
    # --------------------------------------------------------

    n = math.sqrt(
        MU_SUN / a**3
    )

    # --------------------------------------------------------
    # Mean anomaly
    #
    #       M(t) = M0 + n t
    #
    # --------------------------------------------------------

    M = M0 + n * t_days

    # --------------------------------------------------------
    # Solve Kepler's equation
    #
    #       M = E - e sin(E)
    #
    # --------------------------------------------------------

    E = solve_kepler(
        M,
        e
    )

    # --------------------------------------------------------
    # Position in orbital coordinates
    #
    #       x = a(cos(E) - e)
    #
    #       y = a sqrt(1-e^2) sin(E)
    #
    # --------------------------------------------------------

    x = a * (
        math.cos(E) - e
    )

    y = (
        a
        * math.sqrt(1 - e**2)
        * math.sin(E)
    )

    r = np.array([
        x,
        y,
        0.0
    ])

    # --------------------------------------------------------
    # Differentiate Kepler's equation:
    #
    #       M = E - e sin(E)
    #
    # Since
    #
    #       dM/dt = n
    #
    # we get
    #
    #       dE/dt = n / (1 - e cos(E))
    #
    # --------------------------------------------------------

    E_dot = (
        n
        /
        (1 - e * math.cos(E))
    )

    # --------------------------------------------------------
    # Velocity
    #
    #       dx/dt = -a sin(E) E_dot
    #
    #       dy/dt =
    #       a sqrt(1-e^2) cos(E) E_dot
    #
    # --------------------------------------------------------

    vx = (
        -a
        * math.sin(E)
        * E_dot
    )

    vy = (
        a
        * math.sqrt(1 - e**2)
        * math.cos(E)
        * E_dot
    )

    v = np.array([
        vx,
        vy,
        0.0
    ])

    # Rotate the orbital ellipse so its perihelion is
    # not necessarily along the positive x-axis.
    r = rotate_z(
        r,
        varpi
    )

    v = rotate_z(
        v,
        varpi
    )

    return r, v


def planet_state(dt, params):
    """
    Compute the planet state at a given UTC datetime.
    """

    t_days = days_from_j2000(dt)

    return kepler_state(
        t_days,
        params
    )


# ============================================================
# 7. THEORETICAL KEPLER VALUES
# ============================================================

def theoretical_values(params):
    """
    Compute quantities that should remain constant
    for an ideal Kepler ellipse.
    """

    a = params["a"]
    e = params["e"]

    # --------------------------------------------------------
    # Specific orbital energy
    #
    #       E = -mu/(2a)
    #
    # --------------------------------------------------------

    energy = (
        -MU_SUN
        /
        (2 * a)
    )

    # --------------------------------------------------------
    # Specific angular momentum
    #
    #       h = sqrt(mu a (1-e^2))
    #
    # --------------------------------------------------------

    angular_momentum = math.sqrt(
        MU_SUN
        * a
        * (1 - e**2)
    )

    # --------------------------------------------------------
    # Orbital period
    #
    #       P = 2 pi sqrt(a^3 / mu)
    #
    # --------------------------------------------------------

    period = (
        2
        * math.pi
        * math.sqrt(
            a**3 / MU_SUN
        )
    )

    # Minimum and maximum orbital radius
    perihelion = a * (1 - e)
    aphelion = a * (1 + e)

    return {
        "energy": energy,
        "angular_momentum": angular_momentum,
        "period": period,
        "perihelion": perihelion,
        "aphelion": aphelion,
    }


# ============================================================
# 8. COMPUTE NUMERICAL CONSERVATION VALUES
# ============================================================

def verify_planet(
    planet_name,
    params,
    step_days=5
):
    """
    Evaluate energy and angular momentum from 1582 through
    1600 and compare them with exact Kepler values.
    """

    theory = theoretical_values(
        params
    )

    start_date = datetime(
        1582,
        1,
        1,
        tzinfo=timezone.utc
    )

    end_date = datetime(
        1600,
        12,
        31,
        tzinfo=timezone.utc
    )

    current_date = start_date

    rows = []

    while current_date <= end_date:

        # Get position and velocity
        r, v = planet_state(
            current_date,
            params
        )

        radius = np.linalg.norm(r)

        speed = np.linalg.norm(v)

        # ----------------------------------------------------
        # Numerical specific orbital energy
        #
        #       E = 1/2 |v|^2 - mu/r
        #
        # ----------------------------------------------------

        numerical_energy = (
            0.5 * speed**2
            -
            MU_SUN / radius
        )

        # ----------------------------------------------------
        # Numerical specific angular momentum
        #
        #       h = |r x v|
        #
        # ----------------------------------------------------

        h_vector = np.cross(
            r,
            v
        )

        numerical_h = np.linalg.norm(
            h_vector
        )

        # ----------------------------------------------------
        # Relative errors
        # ----------------------------------------------------

        energy_relative_error = (
            abs(
                numerical_energy
                -
                theory["energy"]
            )
            /
            abs(theory["energy"])
        )

        h_relative_error = (
            abs(
                numerical_h
                -
                theory["angular_momentum"]
            )
            /
            abs(theory["angular_momentum"])
        )

        rows.append({
            "date":
                current_date,

            "radius_AU":
                radius,

            "speed_AU_per_day":
                speed,

            "specific_energy":
                numerical_energy,

            "energy_theory":
                theory["energy"],

            "energy_relative_error":
                energy_relative_error,

            "specific_angular_momentum":
                numerical_h,

            "angular_momentum_theory":
                theory["angular_momentum"],

            "angular_momentum_relative_error":
                h_relative_error,
        })

        current_date += timedelta(
            days=step_days
        )

    df = pd.DataFrame(rows)

    # ========================================================
    # Conservation variation across the entire time interval
    # ========================================================

    energy_max = (
        df["specific_energy"].max()
    )

    energy_min = (
        df["specific_energy"].min()
    )

    energy_mean = (
        df["specific_energy"].mean()
    )

    energy_relative_variation = (
        abs(
            energy_max
            -
            energy_min
        )
        /
        abs(energy_mean)
    )

    h_max = (
        df[
            "specific_angular_momentum"
        ].max()
    )

    h_min = (
        df[
            "specific_angular_momentum"
        ].min()
    )

    h_mean = (
        df[
            "specific_angular_momentum"
        ].mean()
    )

    h_relative_variation = (
        abs(
            h_max
            -
            h_min
        )
        /
        abs(h_mean)
    )

    # ========================================================
    # Summary
    # ========================================================

    summary = {
        "planet":
            planet_name,

        "semi_major_axis_AU":
            params["a"],

        "eccentricity":
            params["e"],

        "orbital_period_days":
            theory["period"],

        "perihelion_AU":
            theory["perihelion"],

        "aphelion_AU":
            theory["aphelion"],

        "theoretical_energy":
            theory["energy"],

        "mean_computed_energy":
            energy_mean,

        "max_energy_relative_error":
            df[
                "energy_relative_error"
            ].max(),

        "energy_relative_variation":
            energy_relative_variation,

        "theoretical_angular_momentum":
            theory[
                "angular_momentum"
            ],

        "mean_computed_angular_momentum":
            h_mean,

        "max_angular_momentum_relative_error":
            df[
                "angular_momentum_relative_error"
            ].max(),

        "angular_momentum_relative_variation":
            h_relative_variation,
    }

    return df, summary


# ============================================================
# 9. ENERGY PLOT
# ============================================================

def plot_energy(
    df,
    planet_name
):
    """
    Plot relative error in orbital energy.
    """

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        df["date"],
        df[
            "energy_relative_error"
        ]
    )

    plt.xlabel(
        "Date"
    )

    plt.ylabel(
        "Relative Error"
    )

    plt.title(
        f"{planet_name}: "
        "Specific Orbital Energy Conservation"
    )

    plt.grid(True)

    plt.tight_layout()

    filename = (
        f"{planet_name.lower()}"
        "_energy_verification.png"
    )

    plt.savefig(
        filename,
        dpi=300
    )

    plt.show()


# ============================================================
# 10. ANGULAR MOMENTUM PLOT
# ============================================================

def plot_angular_momentum(
    df,
    planet_name
):
    """
    Plot relative error in angular momentum.
    """

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        df["date"],
        df[
            "angular_momentum_relative_error"
        ]
    )

    plt.xlabel(
        "Date"
    )

    plt.ylabel(
        "Relative Error"
    )

    plt.title(
        f"{planet_name}: "
        "Specific Angular Momentum Conservation"
    )

    plt.grid(True)

    plt.tight_layout()

    filename = (
        f"{planet_name.lower()}"
        "_angular_momentum_verification.png"
    )

    plt.savefig(
        filename,
        dpi=300
    )

    plt.show()


# ============================================================
# 11. OPTIONAL ORBIT SHAPE PLOT
# ============================================================

def plot_orbit(
    planet_name,
    params,
    number_of_points=1000
):
    """
    Additional visual check that the orbit has the expected
    elliptical shape.
    """

    theory = theoretical_values(
        params
    )

    period = theory["period"]

    times = np.linspace(
        0,
        period,
        number_of_points
    )

    x_values = []
    y_values = []

    for t in times:

        r, _ = kepler_state(
            t,
            params
        )

        x_values.append(
            r[0]
        )

        y_values.append(
            r[1]
        )

    plt.figure(
        figsize=(7, 7)
    )

    plt.plot(
        x_values,
        y_values,
        label=f"{planet_name} orbit"
    )

    plt.scatter(
        [0],
        [0],
        marker="o",
        label="Sun"
    )

    plt.xlabel(
        "x (AU)"
    )

    plt.ylabel(
        "y (AU)"
    )

    plt.title(
        f"{planet_name} Noiseless Kepler Orbit"
    )

    plt.axis(
        "equal"
    )

    plt.grid(True)

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        f"{planet_name.lower()}"
        "_orbit_check.png",
        dpi=300
    )

    plt.show()


# ============================================================
# 12. RUN VERIFICATION
# ============================================================

def main():

    print(
        "\n========================================"
    )

    print(
        "HOMEWORK 1.1 ORBIT VERIFICATION"
    )

    print(
        "========================================\n"
    )

    # --------------------------------------------------------
    # EARTH
    # --------------------------------------------------------

    earth_data, earth_summary = (
        verify_planet(
            "Earth",
            EARTH
        )
    )

    # --------------------------------------------------------
    # MARS
    # --------------------------------------------------------

    mars_data, mars_summary = (
        verify_planet(
            "Mars",
            MARS
        )
    )

    # --------------------------------------------------------
    # Summary table
    # --------------------------------------------------------

    summary_table = pd.DataFrame([
        earth_summary,
        mars_summary
    ])

    print(
        summary_table.to_string(
            index=False
        )
    )

    # --------------------------------------------------------
    # Save full numerical results
    # --------------------------------------------------------

    earth_data.to_csv(
        "earth_orbit_verification.csv",
        index=False
    )

    mars_data.to_csv(
        "mars_orbit_verification.csv",
        index=False
    )

    summary_table.to_csv(
        "orbit_verification_summary.csv",
        index=False
    )

    # --------------------------------------------------------
    # Verification plots
    # --------------------------------------------------------

    plot_energy(
        earth_data,
        "Earth"
    )

    plot_angular_momentum(
        earth_data,
        "Earth"
    )

    plot_energy(
        mars_data,
        "Mars"
    )

    plot_angular_momentum(
        mars_data,
        "Mars"
    )

    # Optional geometric verification
    plot_orbit(
        "Earth",
        EARTH
    )

    plot_orbit(
        "Mars",
        MARS
    )

    # --------------------------------------------------------
    # Automated checks
    # --------------------------------------------------------

    tolerance = 1e-12

    for summary in [
        earth_summary,
        mars_summary
    ]:

        planet = summary["planet"]

        energy_ok = (
            summary[
                "energy_relative_variation"
            ]
            < tolerance
        )

        angular_momentum_ok = (
            summary[
                "angular_momentum_relative_variation"
            ]
            < tolerance
        )

        print(
            f"\n{planet}:"
        )

        print(
            "  Energy conserved:",
            energy_ok
        )

        print(
            "  Angular momentum conserved:",
            angular_momentum_ok
        )

        if not energy_ok:
            print(
                "  WARNING: energy variation "
                "is larger than expected."
            )

        if not angular_momentum_ok:
            print(
                "  WARNING: angular momentum "
                "variation is larger than expected."
            )


# ============================================================
# 13. PROGRAM ENTRY POINT
# ============================================================

if __name__ == "__main__":
    main()